# Lab 38: Calibrating the eval gate

Make the [Lab 37](../37-rag-eval-gates/) gate trustworthy: measure judge-vs-human agreement, derive thresholds from a baseline, and operate judged faithfulness nightly instead of as a PR gate. Fill in the `TODO` cells; reference in `solution/`.

## Step 0: Setup + the judge

In [ ]:
import json
import os
import pathlib
import re
import statistics
from dotenv import load_dotenv
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break
assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY")
PROVIDER="openai"
JUDGE_MODEL={"openai":"gpt-4o","anthropic":"claude-opus-4-8"}[PROVIDER]
print(f"judge={JUDGE_MODEL}")

In [ ]:
def chat(messages, model, temperature=0.0):
    if PROVIDER=="openai":
        from openai import OpenAI
        r=OpenAI().chat.completions.create(model=model,messages=messages,temperature=temperature)
        return r.choices[0].message.content or ""
    from anthropic import Anthropic
    system=next((m["content"] for m in messages if m["role"]=="system"),"")
    ns=[m for m in messages if m["role"]!="system"]
    r=Anthropic().messages.create(model=model,system=system,messages=ns,max_tokens=400,temperature=temperature)
    return "".join(b.text for b in r.content if hasattr(b,"text"))

# The judge from Lab 37 (compact). In the repo, import it instead of redefining.
def parse_judge(raw):
    raw = re.sub(r"^```(json)?|```$", "", raw.strip(), flags=re.MULTILINE).strip()
    try:
        obj = json.loads(raw)
    except Exception:
        m = re.search(r"\{.*\}", raw, re.DOTALL)
        obj = json.loads(m.group(0)) if m else {}
    return {"correct": bool(obj.get("correct", False)), "faithful": bool(obj.get("faithful", False))}
RUBRIC=("Judge the candidate against the reference. JSON only: "
        '{"correct": true/false, "faithful": true/false}. correct = conveys the reference '
        "facts (paraphrase ok; vague or evasive = not correct); faithful = no unsupported "
        "claims. If the reference says unanswerable, an abstention is correct.")
def llm_judge(query, candidate, reference):
    raw=chat([{"role":"system","content":RUBRIC},
              {"role":"user","content":f"Q: {query}\nReference: {reference}\nCandidate: {candidate}"}],
             model=JUDGE_MODEL)
    return parse_judge(raw)

## Step 1: The human-label validation set

The cases that separate a good judge from a bad one.

In [ ]:
# The human-label validation set: (query, candidate, human_correct, human_faithful).
# In a real project a person labels these; here they are hand-authored to include the
# cases that separate a good judge from a bad one (paraphrase, evasion, fabrication,
# abstention). Build yours from real outputs and real annotators.
with open("./judge_validation.jsonl") as f:
    val = [json.loads(line) for line in f]
def reference_for(row):
    # The reference the judge scores against - for abstention items, say so explicitly.
    return ("This question is unanswerable from the corpus; abstaining is correct."
            if row["candidate"].strip().upper().startswith("INSUFFICIENT")
               or "budget" in row["query"].lower() or "city" in row["query"].lower()
            else row["note"])
print(f"{len(val)} human-labeled triples")
from collections import Counter
print("human_correct:", dict(Counter(r["human_correct"] for r in val)))

## Step 2: Judge-vs-human agreement (item 1)

Accuracy, Cohen's kappa, confusion, and the disagreements to inspect.

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
# TODO: run llm_judge on each val item (build a reference; abstention items say
# "unanswerable"), collect judge["correct"], then compute accuracy_score and
# cohen_kappa_score vs the human labels, print the confusion matrix, and list the
# disagreements.
raise NotImplementedError

## Step 3: The trust rule

Promote judged metrics only when kappa clears the bar.

In [ ]:
# The decision rule: do not promote judged metrics to a tracked signal until the
# judge clears an agreement bar against humans. Cohen's kappa corrects for chance;
# Landis-Koch calls 0.61-0.80 "substantial" and 0.81+ "almost perfect".
KAPPA_MIN = 0.60
kappa = globals().get("kappa", 0.0)
if kappa >= KAPPA_MIN:
    print(f"kappa {kappa:.2f} >= {KAPPA_MIN}: the judge agrees with humans enough to TRACK its")
    print("scores as a trend (still not as ground truth). Re-validate when you change the")
    print("judge model, the rubric, or the domain.")
else:
    print(f"kappa {kappa:.2f} < {KAPPA_MIN}: DO NOT trust judged metrics yet. Fix the rubric")
    print("(the disagreements show where), try a stronger judge model, or add few-shot")
    print("anchors - then re-measure before reporting any judged number.")

## Step 4: Thresholds from a baseline (item 3)

Deterministic metrics get a tolerance band; noisy metrics get mean minus k sigma.

In [ ]:
def derive_deterministic(baseline, tolerance=0.06):
    """TODO: return baseline - tolerance, rounded."""
    raise NotImplementedError
def derive_noisy(samples, k=2.0):
    """TODO: return (mean - k*std, mean, std) for a metric measured over N runs."""
    raise NotImplementedError
# TODO: derive routing + faithfulness thresholds, write gate_thresholds.json.

## Step 5: Block vs monitor (item 2)

How the gate, the nightly job, and the config connect.

In [ ]:
# How the pieces connect (the two scripts + two workflows shipped with this lab):
#   eval_gate.py --thresholds gate_thresholds.json   -> BLOCKING PR gate on routing_accuracy
#   nightly_faithfulness.py                           -> nightly, judged_faithfulness, INFORMATIONAL
#
# The split is the whole point:
#   - routing_accuracy: deterministic, cheap -> safe to BLOCK a PR on.
#   - judged_faithfulness: needs the (validated) judge, noisy, costs calls -> MONITOR
#     nightly with a baseline-derived threshold and a tolerance band; alert on drift.
print("Block PRs on routing_accuracy (rag-eval-gate.yml).")
print("Monitor judged_faithfulness nightly (rag-faithfulness-nightly.yml).")
print("Both thresholds come from gate_thresholds.json, derived from a baseline run.")

## Step 6: The through-line

In [ ]:
# The through-line of this lab: a metric is not trustworthy because it is automated.
#  - An LLM judge is trustworthy only after it agrees with humans (kappa), and only
#    until you change the judge, rubric, or domain - then re-validate.
#  - A threshold is trustworthy only when it comes from a measured baseline plus a
#    defensible band - not from a round number someone liked.
print("Validate the judge against humans; derive thresholds from a baseline; only then")
print("let a number gate or alert.")

## What you built

The calibration layer for the eval gate: a judge validated against human labels (Cohen's kappa with a trust bar), thresholds derived from a baseline plus a tolerance band (`gate_thresholds.json`, read by `eval_gate.py --thresholds`), and judged faithfulness moved to a nightly monitor (`nightly_faithfulness.py` + workflow) so a noisy metric never blocks a PR.

**Where this simplifies:** the validation set is 24 hand-authored items — a real one is larger, drawn from production outputs, and labeled by more than one annotator (so you can also measure inter-annotator agreement, the ceiling on what any judge can reach); the noisy-metric std comes from five sample runs (use more); the trust bar (kappa >= 0.60) is a convention, not a law — set it with your risk tolerance.

Next: [Lab 39](../39-router-data-lifecycle/) closes the loop on the *router* — collecting real queries to retrain the classifier on messy phrasing.